# 05 · §7.5 Qualitative retrieval panels

Top-k retrieval panels (successes + failures) for Graph-RKD vs the strongest classic baseline. Loads a trained student checkpoint and the test split; green border = same class as query, red = different. Caveat (paper): panels show error *modes*, not ranking quality — that is the metrics' job.

In [ ]:
import os, sys, torch
sys.path.insert(0, os.path.dirname(os.getcwd()))
import torchvision.transforms as T
from metric_common import DATASETS, build_metric_loaders, embed
from model import ConvNextMicro
CKPT = os.environ.get('STUDENT_CKPT', '')   # set to a student_last.pth / best
DATASET = os.environ.get('QUAL_DATASET', 'cub200')
DATA = os.environ.get('DATA', 'data')
assert CKPT and os.path.exists(CKPT), 'defina STUDENT_CKPT p/ um checkpoint treinado'

In [ ]:
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
mean=[0.485,0.456,0.406]; std=[0.229,0.224,0.225]
test_tf = T.Compose([T.Resize((256,256)), T.CenterCrop(224), T.ToTensor(), T.Normalize(mean,std)])
cls, marker = DATASETS[DATASET]
loaders, info = build_metric_loaders(cls, DATA, test_tf, test_tf, 128, 5, 100, 4, 0.2, 0, False)
student = ConvNextMicro(num_classes=2, dims=(24,48,96,192), depths=(1,1,3,1), apply_softmax=False).to(dev)
sd = torch.load(CKPT, map_location=dev); student.load_state_dict(sd.get('best_state') or sd['model']); student.eval()

In [ ]:
import torch.nn.functional as F
embs, labels, imgs = [], [], []
with torch.no_grad():
    for x,y in loaders['test']:
        e,_ = embed(student, x.to(dev), True); embs.append(e.cpu()); labels.append(y); imgs.append(x)
embs=torch.cat(embs); labels=torch.cat(labels); imgs=torch.cat(imgs)
sim = embs @ embs.T; sim.fill_diagonal_(-1)
topk = sim.topk(5, dim=1).indices
print('test embeddings:', embs.shape)

In [ ]:
import matplotlib.pyplot as plt, numpy as np
def show(qs, title):
    fig, ax = plt.subplots(len(qs), 6, figsize=(11, 1.9*len(qs)))
    inv = lambda t: (t*torch.tensor(std)[:,None,None]+torch.tensor(mean)[:,None,None]).clamp(0,1).permute(1,2,0).numpy()
    for r,q in enumerate(qs):
        ax[r,0].imshow(inv(imgs[q])); ax[r,0].set_title('query'); ax[r,0].axis('off')
        for c,j in enumerate(topk[q]):
            a=ax[r,c+1]; a.imshow(inv(imgs[j])); a.axis('off')
            ok = labels[j]==labels[q]
            for s in a.spines.values(): s.set_visible(True); s.set_color('green' if ok else 'red'); s.set_linewidth(3)
    fig.suptitle(title); fig.tight_layout()
# alguns acertos e erros
correct = [i for i in range(len(labels)) if labels[topk[i,0]]==labels[i]]
wrong = [i for i in range(len(labels)) if labels[topk[i,0]]!=labels[i]]
show(correct[:4], 'Graph-RKD student — successes')
show(wrong[:4], 'Graph-RKD student — failures')